<a href="https://colab.research.google.com/github/nattaponm/Teach_Nowcast_Pysteps_TMD/blob/main/notebooks/02_motion_extrapolation_interpolation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2: Motion field, extrapolation และการเพิ่มความละเอียดเวลา
### Optical flow on 15-min TMD data and advection interpolation (15 → 5 min)

⏱️ เวลาเรียนโดยประมาณ **75–90 นาที** | ต้องผ่าน: Notebook 0–1

---
## 🎯 วัตถุประสงค์การเรียนรู้
เมื่อจบ notebook นี้ นิสิตจะสามารถ
1. **คำนวณ motion field** ของพายุด้วย Lucas–Kanade, VET และ DARTS แล้วแปลงเป็นความเร็ว (km/h) และทิศทาง
2. **สร้าง extrapolation nowcast** (Lagrangian persistence) และเปรียบเทียบกับค่าสังเกต
3. **อธิบายข้อจำกัด** ของข้อมูลทุก 15 นาทีต่อการติดตามพายุพาความร้อน
4. **เพิ่มความละเอียดเวลา 15 → 5 นาที** ด้วย advection interpolation และนำไปคำนวณฝนสะสมที่ถูกต้องขึ้น

## 📖 1. แนวคิดและสมการ

### 1.1 Lagrangian persistence และ optical flow (ทบทวน)
$$
\frac{\partial \Psi}{\partial t} + u\,\frac{\partial \Psi}{\partial x} + v\,\frac{\partial \Psi}{\partial y} = 0
$$
- **Lucas–Kanade (LK):** least squares ในหน้าต่างเล็กรอบ feature แล้ว interpolate เป็นฟิลด์ต่อเนื่อง (เร็ว)
- **VET:** หาฟิลด์ทั้งโดเมนที่ทำให้ภาพตรงกันและฟิลด์เรียบ (ช้ากว่า)
- **DARTS:** แก้สมการใน Fourier domain

### 1.2 จาก pixel/time step เป็น km/h
pysteps คืนค่า motion field หน่วย **pixel ต่อ time step** ถ้า pixel = $\Delta x$ km และ time step = $\Delta t$ นาที

$$
|\mathbf{V}|\ (\mathrm{km\,h^{-1}}) = \sqrt{u^2 + v^2}\;\Delta x \;\frac{60}{\Delta t},
\qquad
\text{ทิศที่ไป (องศาจากเหนือ)} = \operatorname{atan2}(u,\,-v)
$$
(ใช้ $-v$ เพราะแถวของ array เพิ่มขึ้นไปทางใต้)

### 1.3 ปัญหาของข้อมูลทุก 15 นาที
เซลล์พายุพาความร้อนมีขนาด 5–20 km และมี **อายุ (lifetime) 30–60 นาที** ในเวลา 15 นาที เซลล์เคลื่อนที่ได้ 10 km หรือมากกว่า และอาจ **เปลี่ยนรูป ก่อตัวใหม่ หรือสลายไป** ทำให้ optical flow "จับคู่" ภาพได้ยาก และฝนสะสมจะเป็นจุดกระโดด

### 1.4 Advection interpolation
สร้างภาพระหว่างกลางด้วยการเลื่อนภาพก่อนหน้าไปข้างหน้า และเลื่อนภาพถัดไปย้อนกลับตาม motion field ($w = \tau/\Delta t$)

$$
\Psi(t_0 + \tau,\mathbf{x}) \approx (1-w)\,\Psi_0(\mathbf{x} - w\,\mathbf{u}) + w\,\Psi_1\big(\mathbf{x} + (1-w)\,\mathbf{u}\big)
$$

## ⚙️ 2. เตรียมสภาพแวดล้อมและข้อมูล

In [ ]:
#@title ⚙️ Setup: ติดตั้ง pysteps และดาวน์โหลดข้อมูล TMD (รันก่อนเสมอ ~2-3 นาทีใน Colab)
import os, sys, warnings
warnings.filterwarnings("ignore")

REPO = "Teach_Nowcast_Pysteps_TMD"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q pysteps==1.21.5 cartopy netCDF4 rasterio geopandas
    if not os.path.exists(REPO):
        !git clone -q --depth 1 https://github.com/nattaponm/{REPO}.git

# หาโฟลเดอร์หลักของ repo (ที่มี utils/ และ data/)
for ROOT in [REPO, ".", ".."]:
    if os.path.exists(os.path.join(ROOT, "utils", "tmd_tools.py")):
        break
ROOT = os.path.abspath(ROOT)
sys.path.insert(0, ROOT)
DATA = os.path.join(ROOT, "data")

import numpy as np
import matplotlib.pyplot as plt
import pysteps
from utils.data_io import load_radar_nc
from utils import mapplot as mp
from utils import tmd_tools as tt
mp.set_style()
mp.BASEMAP.update(provinces=True, rivers=True, lakes=False)   # จังหวัด + แม่น้ำโขง
mp.POINTS[:] = tt.POIS                                         # เรดาร์ เชียงของ เชียงแสน
print("✅ พร้อมใช้งาน | ข้อมูลอยู่ที่:", DATA)

In [ ]:
from pysteps import motion
from pysteps.utils import transformation
from pysteps.extrapolation.semilagrangian import extrapolate

R, meta = load_radar_nc(os.path.join(DATA, "tmd_cri_20200423_rain_1km.nc"))
ts = meta["timestamps"]
DT = meta["accutime"]                         # 15 นาที
KM = meta["xpixelsize"] / 1000                # 1 km
BOX = tt.range_box(meta, 165)
radar_mask = np.isfinite(R[0])

# issue time = 12:00 UTC (index 4): input = 4 ภาพ (11:15–12:00), obs = 8 ภาพ (12:15–14:00)
i0 = 4
R_in, R_obs = R[i0 - 3:i0 + 1], R[i0 + 1:]
t0 = ts[i0]
print("issue time:", mp.tlabel(t0), "| input:", [t.strftime("%H:%M") for t in ts[i0-3:i0+1]],
      "| obs:", len(R_obs), "ภาพ (+15 ถึง +120 นาที)")

dbr, meta_dbr = transformation.dB_transform(R_in, meta, threshold=0.1, zerovalue=-15.0)
dbr[~np.isfinite(dbr)] = -15.0

## 💻 3. Motion field 3 วิธี

In [ ]:
import time
fields, runtime = {}, {}
for name in ["LK", "VET", "DARTS"]:
    tic = time.time()
    if name == "LK":
        fields[name] = motion.get_method("LK")(dbr, verbose=False)
    elif name == "VET":
        fields[name] = motion.get_method("VET")(dbr[-3:], verbose=False)
    else:
        fields[name] = motion.get_method("DARTS")(dbr, N_t=2, verbose=False)
    runtime[name] = time.time() - tic

def speed_dir(V, where):
    """ความเร็ว (km/h) และทิศที่ไป (องศาจากเหนือ) เฉลี่ยในบริเวณ where"""
    u, v = V[0][where], V[1][where]
    spd = np.hypot(u, v) * KM * 60 / DT
    ang = (np.degrees(np.arctan2(np.mean(u), -np.mean(v))) + 360) % 360
    return np.median(spd), ang

storm = np.nan_to_num(R_in[-1]) >= 5          # บริเวณพายุ (≥ 5 mm/h)
for name, V in fields.items():
    s, a = speed_dir(V, storm)
    print(f"{name:6s} {runtime[name]:5.1f} s | ความเร็วมัธยฐานในพายุ {s:5.1f} km/h | ทิศที่ไป {a:5.0f}°")

In [ ]:
fig, axs = mp.map_panels(1, 3, meta, panel_width=3.0, zoom=BOX)
for j, name in enumerate(fields):
    im = mp.radar_map(axs[j], R_in[-1], meta, title=f"{name} ({runtime[name]:.1f} s)", zoom=BOX,
                      gridline_labels=("left", "bottom") if j == 0 else ("bottom",),
                      north=(j == 0), points=(j == 0))
    mp.motion_arrows(axs[j], fields[name], meta, step=22, mask=radar_mask, timestep_min=DT)
mp.add_colorbar(fig, im, axs)
mp.panel_labels(axs)
mp.save_figure(fig, "nb2_fig01_motion_fields")
plt.show()

> 🔍 **สังเกต (สำคัญ!):**
> - **LK** ได้ความเร็วประมาณ 45–50 km/h ไปทางตะวันออกค่อนเหนือ (ENE) ซึ่งตรงกับการเคลื่อนที่ของพายุที่เห็นด้วยตาใน NB1
> - **VET และ DARTS ล้มเหลว** ในกรณีนี้ (ความเร็วใกล้ 0) เพราะ (1) มีฝนเพียง 1–2 % ของพื้นที่ ส่วนใหญ่ของภาพจึงเป็น "ศูนย์ที่ไม่เคลื่อนที่" (2) ใน 15 นาที เซลล์ขนาด 5–10 km เคลื่อนไปไกลกว่าขนาดของมันเอง ทำให้วิธีที่หาความสอดคล้องของ *ทั้งภาพ* จับคู่ไม่ได้
> - บทเรียน: **ต้องตรวจสอบ motion field ด้วยตาเสมอ** ค่า default ที่ใช้ได้ดีกับข้อมูลยุโรปทุก 5 นาที อาจใช้ไม่ได้กับพายุฤดูร้อนในไทยทุก 15 นาที ใน notebook ต่อ ๆ ไปจะใช้ LK

## 🚀 4. Extrapolation nowcast (+15 ถึง +120 นาที)

In [ ]:
V = fields["LK"]
fc = extrapolate(R_in[-1], V, len(R_obs), allow_nonfinite_values=True)
fc[:, ~radar_mask] = np.nan

leads = [0, 1, 3]                                 # +15, +30, +60 นาที
fig, axs = mp.map_panels(2, 3, meta, panel_width=2.9, zoom=BOX)
for j, k in enumerate(leads):
    lt = int((k + 1) * DT)
    im = mp.radar_map(axs[j], R_obs[k], meta, title=f"Observed +{lt} min ({ts[i0+1+k]:%H:%M})", zoom=BOX,
                      gridline_labels=("left",) if j == 0 else (), north=(j == 0), points=(j == 0))
    mp.radar_map(axs[3 + j], fc[k], meta, title=f"Extrapolation +{lt} min", zoom=BOX,
                 gridline_labels=("left", "bottom") if j == 0 else ("bottom",), north=False, points=False)
mp.add_colorbar(fig, im, axs)
mp.panel_labels(axs)
mp.save_figure(fig, "nb2_fig02_extrapolation")
plt.show()

> 🔍 **สังเกต:** ที่ +15 นาที ตำแหน่งยังใกล้เคียง แต่ที่ +60 นาที เซลล์เดิมหลายเซลล์ **สลายไป** และมี **เซลล์ใหม่เกิดขึ้น** ในตำแหน่งที่ extrapolation ไม่มีทางรู้ได้ นี่คือขีดจำกัดพื้นฐานของ nowcasting สำหรับพายุพาความร้อน

## ⏱️ 5. Advection interpolation 15 → 5 นาที และฝนสะสม
ใช้ฟังก์ชัน `tt.advection_interpolate` (โค้ดอยู่ใน `utils/tmd_tools.py` ใช้สมการในข้อ 1.4 และคำนวณ motion field ใหม่ทุกคู่ภาพ)

In [ ]:
def to_dbr(x):
    d, _ = transformation.dB_transform(x, meta, threshold=0.1, zerovalue=-15.0)
    return np.where(np.isfinite(d), d, -15.0)

tic = time.time()
R5, ts5, _ = tt.advection_interpolate(R, meta, n_sub=3, to_dbr=to_dbr)
print(f"15 นาที: {R.shape[0]} ภาพ → 5 นาที: {R5.shape[0]} ภาพ ({time.time()-tic:.1f} s)")

# ฝนสะสม 11:00–14:00 (ใช้กฎสี่เหลี่ยมคางหมู: ภาพแรก/สุดท้ายมีน้ำหนักครึ่งหนึ่ง)
def accumulate(cube, dt_min):
    w = np.ones(len(cube)); w[0] = w[-1] = 0.5
    return np.tensordot(w, np.nan_to_num(cube), axes=1) * dt_min / 60.0

P15 = accumulate(R, 15); P5 = accumulate(R5, 5)
P15[~radar_mask] = np.nan; P5[~radar_mask] = np.nan
print(f"ฝนสะสมสูงสุด: 15 นาที = {np.nanmax(P15):.1f} mm | 5 นาที = {np.nanmax(P5):.1f} mm")
print(f"ค่าเฉลี่ยทั้งรัศมี:  15 นาที = {np.nanmean(P15):.3f} mm | 5 นาที = {np.nanmean(P5):.3f} mm")

In [ ]:
CK_BOX = tt.zoom_box(meta, *tt.CHIANG_KHONG, half_width_km=70)
fig, axs = mp.map_panels(1, 2, meta, panel_width=3.8, zoom=CK_BOX)
im = mp.radar_map(axs[0], P15, meta, ptype="depth", units="mm", title="Every 15 min (original)", zoom=CK_BOX)
mp.radar_map(axs[1], P5, meta, ptype="depth", units="mm", title="Every 5 min (advection-interpolated)",
             zoom=CK_BOX, gridline_labels=("bottom",), north=False)
mp.add_colorbar(fig, im, axs, label="Rainfall 11:00–14:00 UTC (mm)")
mp.panel_labels(axs)
mp.save_figure(fig, "nb2_fig03_accumulation_interpolation")
plt.show()

> 🔍 **สังเกต:** ภาพ (a) มีลักษณะเป็น "ลูกปัด" เรียงตามเส้นทางพายุ เพราะเห็นพายุทุก 15 นาทีเท่านั้น ส่วนภาพ (b) เป็นแถบต่อเนื่อง (swath) ซึ่งสมจริงกว่า และใช้ระบุ **พื้นที่ที่ได้รับผลกระทบจากพายุ** ได้ดีกว่า ซึ่งเป็นประโยชน์ต่อการประเมินความเสียหายจากลูกเห็บ

---
## 🧪 แบบฝึกหัด

### 🧪 แบบฝึกหัด 2.1: พายุเคลื่อนที่เร็วและไปทางไหน?

คำนวณ motion field ด้วย **LK** สำหรับ issue time **ทุกเวลา** ตั้งแต่ index 3 ถึง 12 (ใช้ 4 ภาพก่อนหน้าทุกครั้ง) แล้วหา **ความเร็วมัธยฐาน (km/h)** และ **ทิศที่ไป** ในบริเวณพายุ (≥ 5 mm/h) ของแต่ละเวลา จากนั้นพล็อตกราฟความเร็วกับเวลา

**คำถาม:** ความเร็วของพายุคงที่หรือเปลี่ยนไป? ถ้าข้อมูลมีความถี่ 5 นาทีจะช่วยอย่างไร?

<details>
<summary><b>💡 คำใบ้ (คลิกเพื่อเปิด)</b></summary>

- loop: `for i in range(3, 13):` ใช้ `R[i-3:i+1]`
- แปลง dBR ด้วย `to_dbr(R[i-3:i+1])`
- ใช้ฟังก์ชัน `speed_dir(V, storm_i)` ที่สร้างไว้แล้ว โดย `storm_i = np.nan_to_num(R[i]) >= 5`

</details>

In [ ]:
# ✍️ แบบฝึกหัด 2.1 — เขียนโค้ดของคุณที่นี่ (ลบ # หน้าบรรทัดแล้วเติมส่วนที่เป็น ...)
# rows = []
# for i in range(3, 13):
#     Vi = motion.get_method("LK")(to_dbr(R[i-3:i+1]), verbose=False)
#     s, a = speed_dir(Vi, np.nan_to_num(R[i]) >= 5)
#     rows.append((ts[i], s, a))
# ...

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 2.1 (ลองทำเองก่อน! ดับเบิลคลิกเพื่อดูโค้ด แล้วกด ▶ เพื่อรัน)
import matplotlib.dates as mdates
rows = []
for i in range(3, 13):
    Vi = motion.get_method("LK")(to_dbr(R[i - 3:i + 1]), verbose=False)
    s, a = speed_dir(Vi, np.nan_to_num(R[i]) >= 5)
    rows.append((ts[i], s, a))
    print(f"{mp.tlabel(ts[i])}: {s:5.1f} km/h → {a:4.0f}°")
fig, ax = plt.subplots(figsize=(5.5, 3), layout="constrained")
ax.plot([r[0] for r in rows], [r[1] for r in rows], "o-", color="#1f77b4", ms=4)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
mp.skill_axes(ax, xlabel="Issue time (UTC)", ylabel="Median storm speed (km h$^{-1}$)",
              title="Storm motion from Lucas–Kanade")
mp.save_figure(fig, "nb2_ex21_storm_speed")
plt.show()

### 🧪 แบบฝึกหัด 2.2: วิธีหา motion field ใดให้ nowcast ดีที่สุด?

สร้าง extrapolation nowcast 8 time steps จาก motion field ทั้ง 3 วิธีใน `fields` แล้วคำนวณ **MAE** เทียบกับ `R_obs` ทุก lead time พล็อตกราฟ MAE กับ lead time (นาที)

เพิ่มเส้นอ้างอิง **Eulerian persistence** (ใช้ภาพล่าสุดโดยไม่เลื่อนเลย) ด้วย

**คำถาม:** ทำไม LK ซึ่งได้ motion field ที่ถูกต้องที่สุด กลับได้ MAE แย่กว่าวิธีอื่นที่ lead time ยาว? สิ่งนี้บอกอะไรเกี่ยวกับการเลือกคะแนน verification สำหรับพายุเซลล์เล็ก?

<details>
<summary><b>💡 คำใบ้ (คลิกเพื่อเปิด)</b></summary>

- `fc_m = extrapolate(R_in[-1], fields[name], len(R_obs), allow_nonfinite_values=True)`
- MAE: `np.nanmean(np.abs(fc_m[k] - R_obs[k]))`
- lead time: `(np.arange(len(R_obs)) + 1) * DT`

</details>

In [ ]:
# ✍️ แบบฝึกหัด 2.2 — เขียนโค้ดของคุณที่นี่ (ลบ # หน้าบรรทัดแล้วเติมส่วนที่เป็น ...)
# lead = (np.arange(len(R_obs)) + 1) * DT
# fig, ax = plt.subplots(figsize=(4.5, 3))
# for name in fields:
#     fc_m = extrapolate(...)
#     mae = [... for k in range(len(R_obs))]
#     ax.plot(lead, mae, label=name)
# ax.legend(); plt.show()

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 2.2 (ลองทำเองก่อน! ดับเบิลคลิกเพื่อดูโค้ด แล้วกด ▶ เพื่อรัน)
lead = (np.arange(len(R_obs)) + 1) * DT
colors = {"LK": "#1b9e77", "VET": "#d95f02", "DARTS": "#7570b3"}
fig, ax = plt.subplots(figsize=(4.5, 3), layout="constrained")
for name in fields:
    fc_m = extrapolate(R_in[-1], fields[name], len(R_obs), allow_nonfinite_values=True)
    mae = [np.nanmean(np.abs(fc_m[k][radar_mask] - R_obs[k][radar_mask])) for k in range(len(R_obs))]
    ax.plot(lead, mae, "o-", ms=3, color=colors[name], label=name)
persist = [np.nanmean(np.abs(R_in[-1][radar_mask] - R_obs[k][radar_mask])) for k in range(len(R_obs))]
ax.plot(lead, persist, "k--", lw=1, label="Eulerian persistence")
mp.skill_axes(ax, ylabel="MAE (mm h$^{-1}$)", title="Extrapolation error")
ax.legend(frameon=False, fontsize=7)
mp.save_figure(fig, "nb2_ex22_mae_methods")
plt.show()
# LK ดีที่สุดที่ +15 นาที แต่หลังจากนั้นกลับแย่กว่า "ไม่เลื่อนเลย" (Eulerian persistence)
# เพราะ MAE ลงโทษการพยากรณ์เซลล์ผิดตำแหน่งสองครั้ง (double penalty) และเซลล์เดิมสลายไปแล้ว
# → คะแนนแบบ pixel-by-pixel ไม่เหมาะกับพายุเซลล์เล็ก ต้องใช้ spatial score เช่น FSS (NB3)

### 🧪 แบบฝึกหัด 2.3: Interpolate ละเอียดแค่ไหนจึงพอ?

คำนวณฝนสะสม 11:00–14:00 ด้วย `n_sub = 1` (ไม่ interpolate), `3` (5 นาที), `5` (3 นาที) และ `15` (1 นาที)
แล้วเปรียบเทียบ (ก) **ฝนสะสมสูงสุด** ทั้งโดเมน และ (ข) **ฝนสะสมเฉลี่ยในรัศมี 5 km รอบเชียงของ**

**คำถาม:** ข้อมูลที่ interpolate เป็น 5 นาทีเพียงพอสำหรับงานประเมินพื้นที่ลูกเห็บ/ฝนหนักหรือไม่?

<details>
<summary><b>💡 คำใบ้ (คลิกเพื่อเปิด)</b></summary>

- `Rn, tsn, _ = tt.advection_interpolate(R, meta, n_sub=n, to_dbr=to_dbr)` (ถ้า `n == 1` ใช้ `R` เดิม)
- `accumulate(Rn, 15 / n)`
- ค่าที่เชียงของ: `tt.point_series(P[None], meta, *tt.CHIANG_KHONG, radius_km=5, stat="mean")[0]`

</details>

In [ ]:
# ✍️ แบบฝึกหัด 2.3 — เขียนโค้ดของคุณที่นี่ (ลบ # หน้าบรรทัดแล้วเติมส่วนที่เป็น ...)
# for n in [1, 3, 5, 15]:
#     Rn = R if n == 1 else tt.advection_interpolate(R, meta, n_sub=n, to_dbr=to_dbr)[0]
#     P = accumulate(Rn, 15 / n); P[~radar_mask] = np.nan
#     ck = tt.point_series(P[None], meta, *tt.CHIANG_KHONG, radius_km=5, stat="mean")[0]
#     print(n, np.nanmax(P), ck)

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 2.3 (ลองทำเองก่อน! ดับเบิลคลิกเพื่อดูโค้ด แล้วกด ▶ เพื่อรัน)
print(f"{'Δt':>7s} {'max (mm)':>9s} {'Chiang Khong (mm)':>18s}")
for n in [1, 3, 5, 15]:
    Rn = R if n == 1 else tt.advection_interpolate(R, meta, n_sub=n, to_dbr=to_dbr)[0]
    P = accumulate(Rn, 15 / n)
    P[~radar_mask] = np.nan
    ck = tt.point_series(P[None], meta, *tt.CHIANG_KHONG, radius_km=5, stat="mean")[0]
    print(f"{15/n:5.0f} min {np.nanmax(P):9.1f} {ck:18.1f}")
# ผลเปลี่ยนมากที่สุดระหว่าง 15 → 5 นาที หลังจากนั้นเปลี่ยนน้อย (converge) จึง 5 นาทีเพียงพอสำหรับงานส่วนใหญ่

---
## 📝 สรุป
- Motion field (LK) ของพายุเชียงของมีความเร็วประมาณ **40–50 km/h ไปทางตะวันออกค่อนเหนือ** ส่วน VET/DARTS ใช้ไม่ได้กับข้อมูลชุดนี้
- MAE แบบ pixel-by-pixel ทำให้เข้าใจผิดได้กับพายุเซลล์เล็ก (double penalty)
- Extrapolation ใช้ได้ดีเฉพาะช่วงแรก (+15 นาที) หลังจากนั้นความผิดพลาดหลักมาจาก **การเกิดและสลายของเซลล์**
- ข้อมูลทุก 15 นาทีควร **interpolate เป็น 5 นาที** ก่อนคำนวณฝนสะสมหรือติดตามพายุ (จะใช้อีกครั้งใน NB5)

### ❓ คำถามชวนคิด
1. ถ้าต้องการเตือนภัยลูกเห็บล่วงหน้า 30 นาที ข้อมูลเรดาร์ทุก 15 นาทีมีปัญหาอย่างไรในทางปฏิบัติ?
2. Advection interpolation สร้าง "ข้อมูลใหม่" หรือไม่? ควรระบุอย่างไรในบทความวิจัย?

➡️ **บทถัดไป:** Notebook 3 — deterministic nowcast (Extrapolation, S-PROG, LINDA) และ verification จากหลาย issue time

### 📚 เอกสารอ้างอิง
- Germann, U., & Zawadzki, I. (2002). Scale-dependence of the predictability of precipitation from continental radar images. Part I. *Mon. Wea. Rev.*, 130, 2859–2873.
- Pulkkinen, S., et al. (2019). Pysteps (v1.0). *Geosci. Model Dev.*, 12, 4185–4219.
- ERAD 2022 pysteps course, block 3a: advection-based temporal interpolation.